In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from typing import List
from torch.utils.data import Dataset, DataLoader

import torch
from transformers import AutoImageProcessor, AutoModel

In [ ]:
train_images_dir = "/kaggle/input/exam-photo-avito/train/train"
test_images_dir = "/kaggle/input/exam-photo-avito/test/test"
out_path = "/kaggle/working/"

train = pd.read_parquet("/kaggle/input/table-dataset-exam-avito/train.parquet")
test = pd.read_parquet("/kaggle/input/table-dataset-exam-avito/test.parquet")

In [ ]:
# ускорение для NVIDIA (если cuda)
torch.backends.cudnn.benchmark = True
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2) модель + процессор DINOv2
model_id = "facebook/dinov2-base"
processor = AutoImageProcessor.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id).to(device)
model.eval()

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, image_names, path):
        self.image_names = list(image_names)
        self.path = path
        
    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        name = self.image_names[idx]
        full = os.path.join(self.path, name)
        img = Image.open(full).convert("RGB")
        return img


In [ ]:
def collate(batch):
    """
    функция, которая превращает список отдельных
    объектов из Dataset в один батч тензоров
    """
    return processor(images=batch, return_tensors="pt") # [B, 3, H, W]

In [ ]:
def build_img_emb(images_names, path) -> List[np.ndarray]:
    """
    Функция, которая по списку имён изображений images_names
    и папке path строит эмбеддинги картинок и возвращает матрицу эмбеддингов
    """
    batch_size = 128 if device == "cuda" else 16
    num_workers = 4 if device == "cuda" else 2

    dataset = ImageDataset(images_names, path)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=(device == "cuda"),
        collate_fn=collate,
        persistent_workers=True,
    )
    
    emb_list = []

    with torch.no_grad():
        for inputs in tqdm(loader, desc=f"Embeddings from {os.path.basename(path)}"):
            inputs = {k: v.to(device, non_blocking=True) for k, v in inputs.items()}

            out = model(**inputs)  # out.last_hidden_state: (B, seq_len, hidden)
            # берём CLS-токен как embedding
            feats = out.last_hidden_state[:, 0, :]  # 0 - CLS-токен: (B, hidden), 
            
            emb_list.append(feats.cpu().numpy().astype(np.float32))

    return np.vstack(emb_list) # (N, hidden)

In [ ]:
train_names = train["image_name"].astype(str).values.tolist()
test_names  = test["image_name"].astype(str).values.tolist()

In [ ]:
X_train = build_img_emb(train_names, train_images_dir)
X_test  = build_img_emb(test_names, test_images_dir)

print("X_train:", X_train.shape, "X_test:", X_test.shape)

In [ ]:
# save parquet with image_name
emb_cols = [f"img_emb_{j}" for j in range(X_train.shape[1])]

train_emb_df = pd.DataFrame(X_train, columns=emb_cols)
train_emb_df.insert(0, "image_name", train_names)

test_emb_df = pd.DataFrame(X_test, columns=emb_cols)
test_emb_df.insert(0, "image_name", test_names)

train_path = os.path.join(out_path, "dinov2_train.parquet")
test_path  = os.path.join(out_path, "dinov2_test.parquet")

train_emb_df.to_parquet(train_path, index=False)
test_emb_df.to_parquet(test_path, index=False)

print("Saved:", train_path, test_path)